# Task 1 / OMD（独自モデル部門）ベースライン: 特徴量＋ロジスティック回帰

引用文脈（`[CITE]` が引用位置）と引用先論文の情報から、その引用が**妥当(1)か不適切(0)か**を判定する二値分類タスクです。

Task 1 の配布ベースラインは方向性の異なる2本です:

| notebook | 方向性 | 学べること |
|---|---|---|
| **本 notebook**: 素朴な特徴量＋ロジスティック回帰 | データを見て特徴量を自分で設計する | 特徴量エンジニアリングの基礎 |
| `task1_finetune.ipynb`: 事前学習モデルの fine-tuning | 小型モデルを配布データで学習させる | 教師あり学習の実務一式 |

**データの使い方（この notebook の流れ）**

```
train.jsonl (1,000問・ラベル付き)      → モデルの学習に使う
dev_labeled.jsonl (200問・ラベル付き)  → しきい値・ハイパーパラメータのチューニングに使う
dev_leaderboard.jsonl (100問・ラベルなし) → 予測を出力してリーダーボードへ提出
```

train と dev_labeled は自由に使えます。まずは2段階で提出してみましょう。  
**①ベースライン（数十秒で完了）の直後に提出ファイルを出力し、リーダーボードへ提出してみましょう。**  
**②最終セルで、チューニング済みモデルの提出ファイルを出力**します。

**レギュレーションの要点**（詳細は配布資料を参照）
- 使用できるモデルは1つあたり**パラメータ数 100M 以下**（本 notebook は学習済みモデルを使いません。
  埋め込みモデル等を追加する場合も 100M 以下のものを選んでください）
- **Google Colab Pro で学習が完結**すること。独自環境で学習した重みの持ち込みは不可
- 学習・検証に使えるのは**運営提供データ（そのまま/加工）または自作**


In [1]:
# 必要なライブラリ（初回のみ。ローカルで環境構築済みの場合はスキップ可）
%pip install -q scikit-learn pandas numpy

/Users/kato/Dropbox/Programs/yans/yans-2026-hackathon-admin/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# データの取得（Google Colab 用）: 配布リポジトリをクローンする。
# ローカルで配布リポジトリの中から実行している場合、このセルは何もしません。
![ -d data ] || [ -d ../data ] || git clone -q https://github.com/YANS-official/yans-2026-hackathon

In [3]:
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

# 配布データの場所を自動で探す（./data → ../data の順。環境変数 YANS_DATA_DIR でも指定可能）
# ローカルの Jupyter は notebook のあるフォルダが作業ディレクトリになるため、
# リポジトリ直下の data/ は notebooks/ から見ると ../data になる。
_candidates = ([Path(os.environ["YANS_DATA_DIR"])] if os.environ.get("YANS_DATA_DIR") else []) + [
    Path("data"), Path("../data"), Path("yans-2026-hackathon/data")]
DATA_DIR = next((p for p in _candidates if (p / "train.jsonl").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "配布データ（data/）が見つかりません。notebook と同じ階層か1つ上に data/ を置くか、"
        "環境変数 YANS_DATA_DIR でデータの場所を指定してください")
print(f"データディレクトリ: {DATA_DIR.resolve()}")

# --- 読み込む前に、ファイルの存在とサイズを検査する ---
# Colab へのアップロードが完了する前にセルを実行すると、途中までのファイルを読んで
# UnicodeDecodeError になる。先にサイズを検査して、原因が分かる形で検出する。
MIN_BYTES = {"papers.jsonl": 55_000_000, "train.jsonl": 400_000,
             "dev_labeled.jsonl": 80_000, "dev_leaderboard.jsonl": 40_000}
for name, min_bytes in MIN_BYTES.items():
    path = DATA_DIR / name
    if not path.exists():
        raise FileNotFoundError(
            f"{path} が見つかりません。data/ の場所（DATA_DIR）を確認してください")
    size = path.stat().st_size
    print(f"{name:24s} {size / 1e6:6.2f} MB")
    if size < min_bytes:
        raise RuntimeError(
            f"{name} が本来のサイズ（目安 {min_bytes / 1e6:.1f}MB 以上）より小さく、"
            f"途中で切れています（現在 {size:,} bytes）。アップロードの転送が完了する前に"
            "実行したか、転送が途中で失敗しています。アップロードをやり直し、"
            "ファイルペインの進行表示が消えてからこのセルを再実行してください")

def read_jsonl(path):
    try:
        with open(path, encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]
    except (UnicodeDecodeError, json.JSONDecodeError) as e:
        raise RuntimeError(
            f"{path} の読み込みに失敗しました。ファイルが途中で切れている可能性が高いです。"
            "アップロードをやり直し、転送完了を待ってから再実行してください") from e

train = read_jsonl(DATA_DIR / "train.jsonl")            # 学習用
dev = read_jsonl(DATA_DIR / "dev_labeled.jsonl")        # チューニング用
lb = read_jsonl(DATA_DIR / "dev_leaderboard.jsonl")     # リーダーボード提出対象（ラベルなし）
papers = {p["paper_id"]: p for p in read_jsonl(DATA_DIR / "papers.jsonl")}

print(f"\ntrain={len(train)}  dev_labeled={len(dev)}  dev_leaderboard={len(lb)}  papers={len(papers)}")

FileNotFoundError: 配布データ（data/）が見つかりません。notebook と同じ階層か1つ上に data/ を置くか、環境変数 YANS_DATA_DIR でデータの場所を指定してください

## 1. まずデータを見る

機械学習の第一歩はデータを自分の目で見ることです。1問がどんな形をしているか、ラベルの分布はどうかを確認します。

In [ ]:
ex = train[0]
paper = papers[ex["cited_paper_id"]]
print("== 引用文脈 ==")
print(ex["citation_context"])
print()
print("== 引用先論文 ==")
print("タイトル:", paper["title"])
print("出版年  :", paper["year"])
print("概要    :", paper["abstract"][:200], "...")
print()
print("== ラベル ==", ex["label"], "(1=妥当 / 0=不適切)")

df = pd.DataFrame(train)
print()
print("ラベル分布:", df["label"].value_counts().to_dict())
print("文脈の文字数: 中央値", int(df["citation_context"].str.len().median()),
      "/ 最大", df["citation_context"].str.len().max())

## 2. ベースラインA: 素朴な特徴量＋ロジスティック回帰

「引用が妥当かどうか」は、文脈と論文の**内容が合っているか**で決まります。§1 でデータを眺めると、
たとえば「文脈の話題と論文の概要がかみ合っていない」「文脈は具体的な数値を主張しているのに、
概要にその数値が見当たらない」といった例に気づきます。まずはこうした観察を数値（特徴量）にして、
軽量な分類器（ロジスティック回帰）に学習させます。

ここで使う特徴量（すべて文脈と論文のペアから計算できる）:

1. **文脈とアブストラクトの文字 n-gram 類似度** — 内容が合っていれば語彙が重なるはず
2. **文脈とタイトルの類似度** — 同上
3. **文脈中の数値がアブストラクトに存在する割合** — 数値の主張が論文と合っているかの手がかり

ポイント: **データを眺めて気づいたことを特徴量に変換する、という過程そのものがこのタスクの本質**です。
上の3つは出発点です。自分の観察から新しい特徴量を足していきましょう。


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler

# 文字 n-gram の TF-IDF（日本語は単語分割なしで使える文字 n-gram を用いる）
all_texts = [r["citation_context"] for r in train] + [
    papers[pid]["abstract"] for pid in {r["cited_paper_id"] for r in train}
]
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3), min_df=2).fit(all_texts)

NUM_RE = re.compile(r"\d+(?:\.\d+)?")

def cos(a, b):
    va, vb = tfidf.transform([a]), tfidf.transform([b])
    n = np.sqrt(va.multiply(va).sum()) * np.sqrt(vb.multiply(vb).sum())
    return float(va.multiply(vb).sum() / n) if n > 0 else 0.0

def features(rec):
    ctx = rec["citation_context"]
    p = papers[rec["cited_paper_id"]]
    abst, title = p.get("abstract", ""), p.get("title", "")
    nums = NUM_RE.findall(ctx)
    num_hit = sum(n in abst for n in nums) / len(nums) if nums else 0.5
    return [
        cos(ctx, abst),   # 1. アブストラクト類似度
        cos(ctx, title),  # 2. タイトル類似度
        num_hit,          # 3. 文脈中の数値がアブストラクトに存在する割合
    ]

FEATURE_NAMES = ["abst類似度", "title類似度", "数値一致率"]

X_train = np.array([features(r) for r in train]);  y_train = np.array([int(r["label"]) for r in train])
X_dev = np.array([features(r) for r in dev]);      y_dev = np.array([int(r["label"]) for r in dev])
print("特徴量の形:", X_train.shape)

In [ ]:
scaler = StandardScaler().fit(X_train)
clf_a = LogisticRegression(max_iter=1000).fit(scaler.transform(X_train), y_train)

# train（学習に使ったデータ）と dev_labeled（見ていないデータ）の両方で性能を確認する。
# 2つの差が大きいほど「暗記（過学習）」寄り、両方低ければ「特徴量の情報不足」を疑う。
pred_train_a = clf_a.predict(scaler.transform(X_train))
pred_dev_a = clf_a.predict(scaler.transform(X_dev))
acc_a = accuracy_score(y_dev, pred_dev_a)
print(f"[ベースラインA] train      : Accuracy={accuracy_score(y_train, pred_train_a):.3f}  "
      f"F1={f1_score(y_train, pred_train_a):.3f}")
print(f"[ベースラインA] dev_labeled: Accuracy={acc_a:.3f}  F1={f1_score(y_dev, pred_dev_a):.3f}")
print()
print("特徴量の係数（正=妥当(1)方向に働く）:")
for name, w in zip(FEATURE_NAMES, clf_a.coef_[0]):
    print(f"  {name:10s} {w:+.2f}")

係数を見ると、どの手がかりが判定に効いているかが分かります。

### 2.1 正解・不正解の事例を自分の目で確認する

集計値（Accuracy）だけではモデルの傾向は分かりません。**どんな問題を正しく判定でき、
どんな問題で間違えるのか**を実際に読むことが、次に作る特徴量のヒントになります。

In [ ]:
def show_cases(records, golds, preds, correct, n=3, feats=None):
    """正解(correct=True)または不正解(correct=False)の事例を表示する。"""
    shown = 0
    for i, r in enumerate(records):
        if (preds[i] == golds[i]) != correct:
            continue
        paper = papers[r["cited_paper_id"]]
        print(f"--- {r['id']}  正解={golds[i]}  予測={preds[i]} ---")
        print("文脈:", r["citation_context"])
        print(f"論文: {paper['title']}（{paper['year']}）")
        if feats is not None:
            print("特徴量:", {k: round(float(v), 3) for k, v in zip(FEATURE_NAMES, feats[i])})
        print()
        shown += 1
        if shown >= n:
            return

print("========== 正しく判定できた事例 ==========\n")
show_cases(dev, y_dev, pred_dev_a, correct=True, n=2, feats=X_dev)
print("========== 間違えた事例（ここを読むのが改善の出発点） ==========\n")
show_cases(dev, y_dev, pred_dev_a, correct=False, n=3, feats=X_dev)

間違えた事例に共通するパターンはあるでしょうか？たとえば「引用文脈が具体的すぎて概要と語彙が
重ならない」のように、気づいたことがそのまま**新しい特徴量のアイデア**になります。
文脈と論文情報（タイトル・著者・年・概要・本文）を並べて読み、「人間ならどこを見て
判断するか」を考えてみましょう。

### 2.2 まずリーダーボードに提出してみる

ベースラインAの予測を提出ファイル（1行1問の `{"id": ..., "prediction": 0 or 1}`）として書き出します。
まずこのファイルでリーダーボードへの提出手順をひととおり試しておきましょう。
以降の改善（§3 のチューニングや特徴量追加）が実際にスコアを動かすかを、
いつでも確かめられる状態になります。

In [ ]:
X_lb_a = np.array([features(r) for r in lb])  # dev_leaderboard の特徴量（最終出力でも使う）
pred_lb_a = clf_a.predict(scaler.transform(X_lb_a))

with open("submission_task1_a.jsonl", "w", encoding="utf-8") as f:
    for rec, p in zip(lb, pred_lb_a):
        f.write(json.dumps({"id": rec["id"], "prediction": int(p)}, ensure_ascii=False) + "\n")

print(f"submission_task1_a.jsonl を書き出しました（{len(lb)}行）→ リーダーボードに提出してみましょう")
print("先頭3行:")
for line in open("submission_task1_a.jsonl", encoding="utf-8").readlines()[:3]:
    print(" ", line.strip())

## 3. チューニング: 正則化と特徴量追加の実験場

ベースラインの分類器を **dev_labeled で比較**しながら、正則化の強さ `C` をチューニングして、
提出に使う最終モデルを決めます。特徴量を思いついたら `features()` に追加して、
ここで dev_labeled のスコアが上がるかを確かめる、というループがこの notebook の使い方です。

> **役割分担**: train は「学習」、dev_labeled は「設定の選択」。リーダーボード（dev_leaderboard）は
> 手元では答え合わせができない、最終評価と同じ形式の事前評価です。dev_labeled で上がった工夫がリーダーボードでも
> 上がるかを確かめながら進めましょう。


In [ ]:
results = {"A: 素朴特徴+LR (C=1)": (acc_a, clf_a)}
for C_reg in [0.1, 10.0]:
    clf = LogisticRegression(C=C_reg, max_iter=1000).fit(scaler.transform(X_train), y_train)
    acc = accuracy_score(y_dev, clf.predict(scaler.transform(X_dev)))
    results[f"A: 素朴特徴+LR (C={C_reg})"] = (acc, clf)

print("dev_labeled での比較:")
for name, (acc, _) in results.items():
    print(f"  {acc:.3f}  {name}")

best_name = max(results, key=lambda k: results[k][0])
final_model = results[best_name][1]
print(f"\n最終モデル: {best_name}")

### 3.1 最終モデルの傾向と誤り事例を確認する

最終モデルでも train / dev_labeled の性能差と、誤り事例を確認しておきます。
チューニング前のベースラインから**どの事例が直り、どの事例が新たに壊れたか**も数えてみましょう。


In [ ]:
pred_train_f = final_model.predict(scaler.transform(X_train))
pred_dev_f = final_model.predict(scaler.transform(X_dev))
print(f"[最終モデル] train      : Accuracy={accuracy_score(y_train, pred_train_f):.3f}  "
      f"F1={f1_score(y_train, pred_train_f):.3f}")
print(f"[最終モデル] dev_labeled: Accuracy={accuracy_score(y_dev, pred_dev_f):.3f}  "
      f"F1={f1_score(y_dev, pred_dev_f):.3f}")

fixed = [i for i in range(len(dev)) if pred_dev_a[i] != y_dev[i] and pred_dev_f[i] == y_dev[i]]
broke = [i for i in range(len(dev)) if pred_dev_a[i] == y_dev[i] and pred_dev_f[i] != y_dev[i]]
print(f"\nチューニング前→最終モデル: 直った {len(fixed)}問 / 新たに間違えた {len(broke)}問")

print("\n========== 最終モデルでも間違えている事例（次の改善対象） ==========\n")
show_cases(dev, y_dev, pred_dev_f, correct=False, n=3, feats=X_dev)

## 4. 最終出力: 最良モデルの予測を保存する

最後に、dev_labeled で選んだ最良モデルの予測を書き出します。§2.2 で提出したベースラインの
スコアを、リーダーボードで上回れるか確認しましょう。


In [ ]:
pred_lb = final_model.predict(scaler.transform(X_lb_a))

with open("submission_task1_best.jsonl", "w", encoding="utf-8") as f:
    for rec, pr in zip(lb, pred_lb):
        f.write(json.dumps({"id": rec["id"], "prediction": int(pr)}, ensure_ascii=False) + "\n")

n_diff = int((pred_lb != pred_lb_a).sum())
print(f"submission_task1_best.jsonl を書き出しました（{len(lb)}行 / 予測ラベル分布: "
      f"1が{int(pred_lb.sum())}問, 0が{len(lb) - int(pred_lb.sum())}問）")
print(f"最初の提出とは {n_diff} 問で予測が異なります")

## 5. 改善の方向性

このベースラインは出発点です。たとえば次のような発展が考えられます:

1. **誤り分析**: dev_labeled で間違えた問題を読み、共通するパターンを探して、
   それに対応する特徴量を追加する
2. **本文の活用**: `papers.jsonl` の `tex_content`（LaTeX 本文）をチャンク分割し、文脈と最も近いチャンクの
   類似度を特徴量にする（アブストラクトだけでは情報が足りない問題に効く）。数値を含む主張は、
   本文中の**該当箇所の値**と照合すると判定に有効です
3. **埋め込み特徴の追加**: 100M 以下の公開埋め込みモデル（例: `cl-nagoya/ruri-v3-70m`）で
   文脈と概要の意味的類似度を計算し、特徴量に加える
4. **モデルの学習**: `task1_finetune.ipynb` で小型モデルの fine-tuning に進む。特徴量アプローチと
   fine-tuning の予測を組み合わせる（スタッキング）方法もあります
5. **分類器の変更**: ロジスティック回帰を勾配ブースティング（LightGBM 等）に置き換える

**提出前のチェック**: 推論経路が規定内か（各モデル 100M 以下・商用 API 不使用）、学習が Colab Pro で
再現できるか、を確認してください。
